In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:
%run /Workspace/fmcg_domain/Setup/utilities

In [0]:
bronze_schema,silver_schema,gold_schema
dbutils.widgets.text("catalog","fmcg","Catalog")
dbutils.widgets.text("datasource","customers","Data source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("datasource")

In [0]:
import os 
s3_path = f"s3://databrickssaugat/{data_source}/*.csv"

s3_path

In [0]:
bronze_customers_table = f"{catalog}.{bronze_schema}.{data_source}"
bronze_customers_table

# SILVER PROCESSING

In [0]:
# get bronze data 


silver_customer_df = spark.sql(f"select * from {bronze_customers_table}")

silver_customer_df = silver_customer_df.select(
    col("customer_id").cast("string"),
    col("customer_name").cast("string"),
    col("city").cast("string"),
    col("ingest_timestamp").cast("timestamp"),
    col("file_name").cast("string"),
    col("file_size").cast("long")
)
display(silver_customer_df,10)

In [0]:
# fix leading spaces
silver_customer_df = silver_customer_df.withColumn("customer_name",trim(col("customer_name") )) 

silver_customer_df = silver_customer_df.withColumn("city",trim(col("city") )) 
silver_customer_df = silver_customer_df.withColumn("customer_id",trim(col("customer_id") ))
 

In [0]:
display(silver_customer_df,10)

In [0]:
cities = silver_customer_df.select("city").distinct().toPandas()["city"].tolist()

In [0]:
cities

In [0]:
from itertools import chain

city_mappings = {'Bengaluru':'Bengalore',
                'Hyderabad':'Hyderabad',
                'New Delhi':'New Delhi',
                'Bengalore':'Bengalore',
                'Hyderabadd':'Hyderabad',
                'Hyderbad':'Hyderabad',
                'NewDelhee':'New Delhi',
                'NewDelhi':'New Delhi',
                'Bengaluruu':'Bengalore',
                'NewDheli':'New Delhi',
                }

mapping_expr = create_map(
    [lit(x) for x in chain(*city_mappings.items())]
)

silver_customer_df = silver_customer_df.withColumn("city", mapping_expr[col("city")])

In [0]:
silver_customer_df = silver_customer_df.withColumn("customer_name",
                                                   
                                                   when(col("customer_name")=="UNKNOWN",None)
                                                   .otherwise(initcap(col("customer_name")))


                                                   )

In [0]:
display(silver_customer_df)

In [0]:
silver_customer_df.filter(col("city").isNull()).display()

In [0]:
data_fixtures = {
"789403":"Hyderabad",
"789420":"New Delhi",
"789521":"Hyderabad",
"789603":"New Delhi",
"789603":"New Delhi"
}

fix_df = spark.createDataFrame(data_fixtures.items(), ["customer_id", "city_fix"])

silver_customer_df = silver_customer_df.join(fix_df, "customer_id", "left").withColumn("city",coalesce(col("city"),col("city_fix"))).drop(col("city_fix"))

display(silver_customer_df)


In [0]:
silver_customer_df = silver_customer_df.withColumn(
    "customer_code",
    col("customer_id")
)


In [0]:
silver_customer_df.columns

In [0]:
silver_customer_df = silver_customer_df.drop(col("cusotmer_id"))

In [0]:
silver_customer_df = silver_customer_df.withColumn(
    "customer" , 
    when(
        col("city").isNotNull(), 
        concat_ws("-", col("customer_name"), col("city"))
    ).otherwise(
        concat_ws("-", col("customer_name"), lit("Unknown"))
    )
)


In [0]:
silver_customer_df = silver_customer_df.withColumn(
    "market",
    lit("India")
).withColumn(
    "platform",
    lit("Sports Bar")
).withColumn(
    "channel",
    lit("Acquisition")
)

display(silver_customer_df)

In [0]:
silver_customer_table = f"{catalog}.{silver_schema}.{data_source}"
silver_customer_df.write \
  .mode("overwrite") \
  .format("delta") \
  .option("delta.enableChangeDataFeed", "true")\
  .saveAsTable(silver_customer_table) 